# 04 — Extractor

**Module notebook — definitions only.**

Pulls structured info (action items, decisions, open questions) out of the transcript.

Depends on: `get_llm()` (loaded in `00_llm_config.ipynb`).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


In [ ]:
def build_chain(system_prompt: str, language: str = "english"):
    llm = get_llm()
    full_prompt = system_prompt + "\n\n" + output_language_instruction(language)
    return (
        RunnablePassthrough() | RunnableLambda(lambda x: {"text": x}) | ChatPromptTemplate.from_messages([
            ("system", full_prompt),
            ("human", "{text}"),
        ]) | llm | StrOutputParser()
    )


def _fit_to_context_window(transcript: str, label: str) -> str:
    """These extraction calls already send the whole transcript in one shot
    (no chunking) — safe under the LLM\'s 262K token context window
    (TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT, from 00_llm_config.ipynb), which in
    practice covers tens of hours of speech. If a transcript somehow exceeds
    that ceiling, truncate with a clear, visible warning rather than silently
    sending an oversized request and risking a provider-side truncation or
    error with no explanation."""
    if len(transcript) <= TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT:
        return transcript

    print(
        f"{label}: transcript ({len(transcript):,} chars) exceeds the "
        f"single-call limit ({TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT:,} chars) — "
        "truncating to the first part of the transcript for this extraction. "
        "(This is well beyond this project\'s intended video-length range.)"
    )
    return transcript[:TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT]


In [ ]:
def extract_action_items(transcript: str, language: str = "english") -> str:
    transcript = _fit_to_context_window(transcript, "extract_action_items")
    chain = build_chain(
        "You are an expert meeting analyst. From the meeting transcript, "
        "extract all action items. For each provide:\n"
        "- Task description\n"
        "- Owner (who is responsible)\n"
        "- Deadline (if mentioned, else write \'Not specified\')\n\n"
        "Format as a numbered list. If none found say \'No action items found.\'",
        language=language,
    )
    return chain.invoke(transcript)


def extract_key_decisions(transcript: str, language: str = "english") -> str:
    transcript = _fit_to_context_window(transcript, "extract_key_decisions")
    chain = build_chain(
        "You are an expert meeting analyst. From the meeting transcript, "
        "extract all key decisions made. Format as a numbered list. "
        "If none found say \'No key decisions found.\'",
        language=language,
    )
    return chain.invoke(transcript)


def extract_questions(transcript: str, language: str = "english") -> str:
    transcript = _fit_to_context_window(transcript, "extract_questions")
    chain = build_chain(
        "From the meeting transcript, extract all unresolved questions "
        "or topics needing follow-up. Format as a numbered list. "
        "If none found say \'No open questions found.\'",
        language=language,
    )
    return chain.invoke(transcript)
